# Molecular Benchmark Results

This notebook reads the benchmark CSV outputs, compares random vs scaffold splits, and writes prediction plots from the saved validation/test predictions.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.visualize import (
    plot_predicted_vs_actual,
    plot_random_vs_scaffold_summary,
    plot_residual_distribution,
    save_prediction_figures,
)

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

benchmark_results = pd.read_csv(RESULTS_DIR / "benchmark_results.csv")
benchmark_summary = pd.read_csv(RESULTS_DIR / "benchmark_summary.csv")
predictions = pd.read_csv(RESULTS_DIR / "predictions.csv")

benchmark_results.shape, benchmark_summary.shape, predictions.shape

((300, 14), (60, 17), (52875, 11))

## Random vs Scaffold Summary

The table and chart below compare random and scaffold splits for the Ridge descriptor baseline across ESOL and FreeSolv.

In [2]:
summary_view = benchmark_summary.loc[
    (benchmark_summary["feature_type"] == "descriptors")
    & (benchmark_summary["model_key"] == "ridge"),
    [
        "dataset",
        "feature_type",
        "model_key",
        "split_type",
        "n_seeds",
        "test_rmse_mean",
        "test_rmse_std",
        "test_mae_mean",
        "test_mae_std",
        "test_r2_mean",
        "test_r2_std",
    ],
].sort_values(["dataset", "split_type"])

display(summary_view)

ax = plot_random_vs_scaffold_summary(
    benchmark_summary,
    metric="test_rmse_mean",
    feature_type="descriptors",
    model_key="ridge",
)
ax.figure.tight_layout()
plt.show()

,dataset,feature_type,model_key,split_type,n_seeds,test_rmse_mean,test_rmse_std,test_mae_mean,test_mae_std,test_r2_mean,test_r2_std
16,esol,descriptors,ridge,random,5,0.908222,0.043570,0.711650,0.042825,0.819139,0.028796
17,esol,descriptors,ridge,scaffold,5,1.224664,0.120914,0.873912,0.058907,0.662384,0.054563
46,freesolv,descriptors,ridge,random,5,0.410580,0.017814,0.312471,0.017380,0.798766,0.048596
47,freesolv,descriptors,ridge,scaffold,5,0.574022,0.038067,0.425279,0.021127,0.687181,0.112107


/var/folders/76/n9t9gpfd0fv2h7d9d4ypy0380000gn/T/ipykernel_2912/344967214.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Prediction Plots

The plots use held-out test rows from seed 0 for the Ridge descriptor random-split baseline. Each title includes dataset, model, and split.

In [3]:
plot_configs = [
    {
        "dataset": "esol",
        "feature_type": "descriptors",
        "model_key": "ridge",
        "split_type": "random",
        "seed": 0,
        "split": "test",
    },
    {
        "dataset": "freesolv",
        "feature_type": "descriptors",
        "model_key": "ridge",
        "split_type": "random",
        "seed": 0,
        "split": "test",
    },
]

for config in plot_configs:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    plot_predicted_vs_actual(predictions, ax=axes[0], **config)
    plot_residual_distribution(predictions, ax=axes[1], **config)
    fig.tight_layout()
    plt.show()

saved_figures = save_prediction_figures(
    predictions,
    output_dir=FIGURES_DIR,
    combinations=plot_configs,
)
saved_figures

/var/folders/76/n9t9gpfd0fv2h7d9d4ypy0380000gn/T/ipykernel_2912/2703572148.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/76/n9t9gpfd0fv2h7d9d4ypy0380000gn/T/ipykernel_2912/2703572148.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


{'predicted_vs_actual': ['/Users/yingzhou/Documents/Molecular/results/figures/predicted_vs_actual_esol_ridge_random.png',
  '/Users/yingzhou/Documents/Molecular/results/figures/predicted_vs_actual_freesolv_ridge_random.png'],
 'residuals': ['/Users/yingzhou/Documents/Molecular/results/figures/residuals_esol_ridge_random.png',
  '/Users/yingzhou/Documents/Molecular/results/figures/residuals_freesolv_ridge_random.png']}